# Imports

In [1]:
from pathlib import Path
print(Path.cwd())

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent)) # problem with dependency resolution (e.g. custom_builder)without this

/home/bernard/Projects/dic/Week 3


In [2]:
# Use delta features if needed (DeltaTable, etc.)
from delta import *
from custom_builder import builder
from log import *

# use the existing preconfigured builder to create the Spark session.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f'current database: {spark.catalog.currentDatabase()}')
print(f'spark tables: {spark.catalog.listTables()}')

from pyspark.sql import functions as F

import json

:: loading settings :: url = jar:file:/home/bernard/Projects/dic/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/bernard/.ivy2.5.2/cache
The jars for the packages stored in: /home/bernard/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5d060d3c-b542-4f54-af73-555a494b78c8;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#d

current database: default
spark tables: [Table(name='air_quality', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='integrated_taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_zone_lookup', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='weather', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False)]


# Load dataset file

In [3]:
from pyspark.sql.types import StringType, IntegerType, LongType, DoubleType, TimestampType, TimestampNTZType, FloatType, ShortType, DecimalType


# load data/air_quality/hourly_88101_2024.csv into a DataFrame
air_quality_df = spark.read.csv("../data/air_quality/hourly_88101_2024.csv", header=True, inferSchema=False)

# cast necessary columns to appropriate data types
for field in air_quality_df.schema.fields:
    if field.name == "Sample Measurement" or field.name == "MDL":
        air_quality_df = air_quality_df.withColumn(field.name, F.col(field.name).cast(DoubleType()))

air_quality_df.show(5)
air_quality_df.printSchema()

+----------+-----------+--------+--------------+---+---------+----------+-----+--------------------+----------+----------+----------+--------+------------------+--------------------+---+-----------+---------+-----------+-----------+--------------------+----------+-----------+-------------------+
|State Code|County Code|Site Num|Parameter Code|POC| Latitude| Longitude|Datum|      Parameter Name|Date Local|Time Local|  Date GMT|Time GMT|Sample Measurement|    Units of Measure|MDL|Uncertainty|Qualifier|Method Type|Method Code|         Method Name|State Name|County Name|Date of Last Change|
+----------+-----------+--------+--------------+---+---------+----------+-----+--------------------+----------+----------+----------+--------+------------------+--------------------+---+-----------+---------+-----------+-----------+--------------------+----------+-----------+-------------------+
|        01|        003|    0010|         88101|  3|30.497478|-87.880258|NAD83|PM2.5 - Local Con...|2024-01-0

# Summary of dataset

Note that negative values do occur in original dataset so synthetic also having some is to be expected

In [4]:
# summary of dataset
air_quality_df.describe().show()

26/09/25 21:41:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 10:>                                                         (0 + 1) / 1]

+-------+-----------------+-----------------+------------------+--------------+------------------+-----------------+------------------+-------+--------------------+----------+----------+----------+--------+------------------+--------------------+------------------+-----------+------------------+-----------+------------------+--------------------+----------+-----------+-------------------+
|summary|       State Code|      County Code|          Site Num|Parameter Code|               POC|         Latitude|         Longitude|  Datum|      Parameter Name|Date Local|Time Local|  Date GMT|Time GMT|Sample Measurement|    Units of Measure|               MDL|Uncertainty|         Qualifier|Method Type|       Method Code|         Method Name|State Name|County Name|Date of Last Change|
+-------+-----------------+-----------------+------------------+--------------+------------------+-----------------+------------------+-------+--------------------+----------+----------+----------+--------+----------

# Generate rows

Decimal values are handled with column values from normal distribution (mean, std) delimited by observed real range  

String values are handled with categorical distribution

Integer values are handled with categorical distribution

In [5]:
# get latest Date Local value
latest_date = air_quality_df.select(F.max("Date Local")).first()[0]
print(latest_date)

[Stage 11:===========================================>            (14 + 4) / 18]

2024-12-31


In [6]:
import random
import string
from datetime import timedelta
from pyspark.sql.window import Window

number_of_rows = int(0.1 * air_quality_df.count())


existing_values_and_stats = {}

unique = (
    air_quality_df.select("State Code", "County Code", "Site Num", "Parameter Code", "POC", "Latitude", "Longitude", "Datum", "Parameter Name", "Units of Measure", "Uncertainty", "Qualifier", "Method Type", "Method Code", "Method Name", "State Name", "County Name")
      .distinct()
      .withColumn(
          "idx",
          F.row_number().over(Window.orderBy(F.lit(1))) - 1
      )
)

n_unique = unique.count()



# find the existing values and stats for each column in air_quality_df
for field in air_quality_df.schema.fields:
    if field.name == "Sample Measurement" or field.name == "MDL":
        

        # calculate mean and std of the column
        mean_std = air_quality_df.select(F.mean(field.name), F.stddev(field.name)).first()
        mean = mean_std[0] if mean_std else None
        stddev = mean_std[1] if mean_std else None
        if mean is None or stddev is None:
            # error
            raise ValueError(f"Cannot calculate mean and stddev for column {field.name}")
        else:
            existing_values_and_stats[field.name] = {
                "Mean": mean,
                "Stddev": stddev,
                "max": air_quality_df.select(F.max(field.name)).first()[0],
                "min": air_quality_df.select(F.min(field.name)).first()[0]
            }
    
print("Computed existing values and stats for relevant columns in air_quality_df")
    


[Stage 38:===========================================>            (14 + 4) / 18]

Computed existing values and stats for relevant columns in air_quality_df


In [7]:

from pyspark.sql import functions as F


def uniform_from_values(values, data_type, seed=None):
    n = len(values)

    return F.element_at(
        F.array(*[F.lit(v).cast(data_type) for v in values]),
        (F.rand(seed) * n).cast("int") + 1,
    )

def normal_from_stats(existing_values_and_stats, column_name, seed=None):
    mean = existing_values_and_stats[column_name]["Mean"]
    stddev = existing_values_and_stats[column_name]["Stddev"]
    min_value = existing_values_and_stats[column_name]["min"]
    max_value = existing_values_and_stats[column_name]["max"]

    return F.least(
        F.lit(max_value),
        F.greatest(
            F.lit(min_value),
            F.lit(mean) + F.lit(stddev) * F.randn(seed)
        )
    )


seed = 42
generated_df = (
    spark.range(number_of_rows)
    # Timestamp columns: generate timestamps based on the latest date in the dataset
    .withColumn(
        "ts",
        F.expr(f"timestamp('{latest_date}') + id * interval 1 hour")
    )
    .withColumn(
        "Date Local",
        F.to_date(F.col("ts") - F.expr("INTERVAL 6 HOURS"))
    )
    .withColumn(
        "Time Local",
        F.date_format(
            F.col("ts") - F.expr("INTERVAL 6 HOURS"),
            "HH:mm"
        )
    )
    .withColumn(
        "Date GMT",
        F.to_date(F.col("ts"))
    )
    .withColumn(
        "Time GMT",
        F.date_format(
            F.col("ts"),
            "HH:mm"
        )
    )
    .withColumn(
        "Date of Last Change",
        F.current_date()
    )
    # Numerical columns: 
    .withColumn(
        "Sample Measurement",
        normal_from_stats(existing_values_and_stats, "Sample Measurement", seed)
    )
    .withColumn(
        "MDL",
        normal_from_stats(existing_values_and_stats, "MDL", seed)
    )
    # new columns
    .withColumn(
        "aqi",
        # random integer between 0 and 500
        (F.rand(seed) * 500).cast("int")
    )
    # Necessary realistic columns: pick random existing combination
    .withColumn("idx", (F.rand(seed) * n_unique).cast("long"))
    .join(unique, "idx")
)

# drop the id column generated by spark.range
generated_df = generated_df.drop("id")
generated_df = generated_df.drop("ts")
generated_df = generated_df.drop("idx")

# reorder the columns to match the original air_quality_df, also make sure to get the new columns
generated_df = generated_df.select(air_quality_df.columns + ["aqi"])



generated_df.show(30)
generated_df.printSchema()

26/09/25 21:45:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/25 21:45:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/25 21:45:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/25 21:45:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/25 21:45:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/25 21:45:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/25 2

+----------+-----------+--------+--------------+---+---------+-----------+-----+--------------------+----------+----------+----------+--------+-------------------+--------------------+-------------------+-----------+---------+-----------+-----------+--------------------+--------------+------------+-------------------+---+
|State Code|County Code|Site Num|Parameter Code|POC| Latitude|  Longitude|Datum|      Parameter Name|Date Local|Time Local|  Date GMT|Time GMT| Sample Measurement|    Units of Measure|                MDL|Uncertainty|Qualifier|Method Type|Method Code|         Method Name|    State Name| County Name|Date of Last Change|aqi|
+----------+-----------+--------+--------------+---+---------+-----------+-----+--------------------+----------+----------+----------+--------+-------------------+--------------------+-------------------+-----------+---------+-----------+-----------+--------------------+--------------+------------+-------------------+---+
|        21|        111|    

In [8]:
generated_df.describe().show()

26/09/25 21:45:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/25 21:45:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/25 21:45:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/25 21:45:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/25 21:45:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/25 21:45:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/25 2

+-------+------------------+-----------------+------------------+--------------------+------------------+-----------------+------------------+------+--------------------+----------+--------+------------------+--------------------+------------------+-----------+------------------+-----------+------------------+--------------------+----------+-----------+------------------+
|summary|        State Code|      County Code|          Site Num|      Parameter Code|               POC|         Latitude|         Longitude| Datum|      Parameter Name|Time Local|Time GMT|Sample Measurement|    Units of Measure|               MDL|Uncertainty|         Qualifier|Method Type|       Method Code|         Method Name|State Name|County Name|               aqi|
+-------+------------------+-----------------+------------------+--------------------+------------------+-----------------+------------------+------+--------------------+----------+--------+------------------+--------------------+------------------+-

# Write dataframe to csv file
Note: crc files are only metadata

In [9]:
# save generated_df to single csv file
generated_df.coalesce(1).write.mode("overwrite").option("header", "true").csv("../updates/air_quality_update")

26/09/25 21:45:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/25 21:45:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/25 21:45:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/25 21:45:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/25 21:46:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/25 21:46:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/25 2